In [1]:
## **課題１**

# 授業中に紹介したSKABデータについてXGBoostを用いてモデル構築を行い，
# RandomForestの変数重要度，SHAP値と比較してどのような違いがあるか検証を行え．
# また，可能であればSVMなど他の予測モデルで予測を行った結果と比較を行い，
# 予測に効く変数が検討したモデルでどのように異なるか考察を行え．

## **課題２**

# 自分の興味のあるデータ（対象は何でも良いが回帰の問題をおすすめする：SHAPバージョンによるトラブルを避けるため）について今回と同様のRandomForestやSHAP値による解析を行い，
# 対象のデータの予測に効く変数がどのようなものか，またそこからどのようなことが考察できるのか（どのような現象が発生しているか？など）について考察を行え

In [6]:
%pip install xgboost
%pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/549.1 kB ? eta -:--:--
   ---------------------------------------- 549.1/549.1 kB 11.2 MB/s  0:00:00

   ------------- -------------------------- 1/3 [cloudpickle]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   ---------------------------------------- 3/3 [shap]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.inspection import permutation_importance

#sklearnライブラリのtreeから決定木の読み込み
from sklearn.tree import DecisionTreeClassifier
#決定木構造を書き出すための関数の読み込み
# from sklearn.tree import export_graphviz
#決定木を表示するためのライブラリを読み込む
# import graphviz
from sklearn.tree import plot_tree

In [25]:
# データの読み込み
df = pd.read_csv('data/valve1.csv',)

df.describe().T

,count,mean,std,min,25%,50%,75%,max
Accelerometer1RMS,18160.0,0.027338,0.000527,0.025553,0.026971,0.027346,0.027722,0.029031
Accelerometer2RMS,18160.0,0.040546,0.001168,0.037339,0.039761,0.040461,0.041192,0.053439
Current,18160.0,0.974279,0.273464,0.355411,0.739579,0.986835,1.202650,1.662610
Pressure,18160.0,0.063758,0.258584,-1.257000,0.054711,0.054711,0.054711,1.694350
Temperature,18160.0,70.207720,2.608980,65.188700,68.630525,69.765300,71.075925,79.889100
Thermocouple,18160.0,25.029108,0.447720,24.418700,24.674200,24.866800,25.423625,26.104400
Voltage,18160.0,230.710674,10.817408,203.135000,224.785750,230.959000,236.880500,255.324000
Volume Flow RateRMS,18160.0,31.284849,1.991965,22.000000,31.001000,32.000000,32.012600,33.969400
anomaly,18160.0,0.347412,0.476161,0.000000,0.000000,0.000000,1.000000,1.000000
changepoint,18160.0,0.003469,0.058799,0.000000,0.000000,0.000000,0.000000,1.000000


In [26]:
# 欠損値の確認
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18160 entries, 0 to 18159
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   datetime             18160 non-null  object 
 1   Accelerometer1RMS    18160 non-null  float64
 2   Accelerometer2RMS    18160 non-null  float64
 3   Current              18160 non-null  float64
 4   Pressure             18160 non-null  float64
 5   Temperature          18160 non-null  float64
 6   Thermocouple         18160 non-null  float64
 7   Voltage              18160 non-null  float64
 8   Volume Flow RateRMS  18160 non-null  float64
 9   anomaly              18160 non-null  float64
 10  changepoint          18160 non-null  float64
dtypes: float64(10), object(1)
memory usage: 1.5+ MB


In [27]:
df = df.drop(["datetime", "changepoint"], axis=1)
df

,Accelerometer1RMS,Accelerometer2RMS,Current,Pressure,Temperature,Thermocouple,Voltage,Volume Flow RateRMS,anomaly
0,0.026588,0.040111,1.330200,0.054711,79.3366,26.0199,233.062,32.0000,0.0
1,0.026170,0.040453,1.353990,0.382638,79.5158,26.0258,236.040,32.0000,0.0
2,0.026199,0.039419,1.540060,0.710565,79.3756,26.0265,251.380,32.0000,0.0
3,0.026027,0.039641,1.334580,0.382638,79.6097,26.0393,234.392,32.0000,0.0
4,0.026290,0.040273,1.078510,-0.273216,79.6109,26.0420,225.342,32.0000,0.0
...,...,...,...,...,...,...,...,...,...
18155,0.027605,0.039760,0.622996,0.382638,68.4247,24.4370,230.358,32.9673,0.0
18156,0.027286,0.039613,0.600692,0.054711,68.0598,24.4356,231.373,32.0000,0.0
18157,0.027203,0.041440,0.450323,0.054711,68.1836,24.4379,210.605,32.0337,0.0
18158,0.027180,0.041831,0.855527,0.382638,68.2250,24.4310,229.566,32.9673,0.0


In [29]:
# 説明変数と目的変数の分割
df_x = df.drop('anomaly', axis=1)
df_y = df.anomaly

In [ ]:
# 決定木を学習させる
clf = DecisionTreeClassifier(random_state=0)
clf = clf.fit(df_x, df_y)

# 学習結果を出力
display(graphviz.Source(export_graphviz(clf, feature_names=df_x.columns, impurity=True, filled=True)))

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [ ]:
# 学習データとテストデータの分割
X_train, X_test, y_train, y_test = train_test_split(df_x, df_y, test_size=0.2, random_state=42)

In [ ]:
# モデルの構築
model = SVC(kernel='rbf', probability=True)
model.fit(X_train, y_train)

# SHAP値の計算
model = shap.KernelExplainer(model.predict_proba)
shap_values = explainer.shap_values(X_test, check_additivity=False)

# SHAP値の可視化